# BRISC — DRL: Brain Tumour MRI Refinement (Colab Edition)

**Colab-curated.** Due to time constraints this notebook only exposes **DuelingDDQN** and **DDPG**
(the two algorithms going forward) — DQN and the experimental contour-refinement paradigm are left
out of the picker but still exist in the configs if you need them later.

| Variable | Options |
|---|---|
| `TUMOR_TYPE` | `'all'` (combined) · `'glioma'` · `'meningioma'` · `'pituitary'` |
| `AGENT_NAME` | `'DuelingDDQN'` · `'DDPG'` |

**Paradigm:** both agents do **local mask refinement** (24-action `SegmentationEnv` for
DuelingDDQN, 3-D continuous morphology+shift for DDPG) on top of a frozen U-Net baseline mask.

**BRISC baseline:** val Dice 0.8290 · test Dice 0.8351 · HD95 8.36 px

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free tier) or better.
2. Upload your `BRISC` (or `brisc2025`) dataset folder and `brisc_best.pt` baseline checkpoint
   to Google Drive (see §0 for the exact paths this notebook looks for).
3. Run cells top-to-bottom. §3 (dry-run) is optional but recommended before §4 (full training).


## 0 · Mount Drive + clone repo + install deps

Expects your Google Drive laid out as:
```
MyDrive/iteris_data/BRISC/...            # dataset (or 'brisc2025')
MyDrive/iteris_data/brisc_best.pt        # baseline checkpoint
MyDrive/iteris_checkpoints/              # DRL checkpoints get written here
```
Adjust `DRIVE_DATA_ROOT` / `DRIVE_CKPT_DIR` below if you used different folder names.


In [ ]:
import subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# Adjust these if your Drive layout differs
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/iteris_data')
DRIVE_CKPT_DIR  = Path('/content/drive/MyDrive/iteris_checkpoints')
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL  = 'https://github.com/Anfaal26/iteris.git'
PKG_ROOT  = Path('/content/iteris')
if not (PKG_ROOT / 'iteris' / '__init__.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PKG_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PKG_ROOT), 'pull'], check=True)

subprocess.run(['pip', 'install', '-r', str(PKG_ROOT / 'requirements.txt'),
                '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))

import torch
print(f'Package at: {PKG_ROOT}')
print(f'GPU available: {torch.cuda.is_available()}  '
      f'({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only - go to Runtime > Change runtime type > GPU"})')


## 1 · Configure — pick tumour subtype + algorithm here

In [ ]:
# ==============================================================================
TUMOR_TYPE = 'all'          # 'all' | 'glioma' | 'meningioma' | 'pituitary'
AGENT_NAME = 'DuelingDDQN'  # 'DuelingDDQN' | 'DDPG'
# ==============================================================================

assert AGENT_NAME in ('DuelingDDQN', 'DDPG'), (
    "This Colab edition only supports 'DuelingDDQN' or 'DDPG' - "
    "DQN and DuelingDDQN_CONTOUR still exist in the configs but aren't wired up here."
)

_CONFIG_MAP = {
    'all':        'brisc_drl_tumor.yaml',
    'glioma':     'brisc_drl_glioma.yaml',
    'meningioma': 'brisc_drl_meningioma.yaml',
    'pituitary':  'brisc_drl_pituitary.yaml',
}
assert TUMOR_TYPE in _CONFIG_MAP, f"TUMOR_TYPE must be one of {list(_CONFIG_MAP)}"

from iteris.config import load_drl_class_config, resolve_agent_config, load_config
from iteris.utils  import get_device

cfg_full     = load_drl_class_config(str(PKG_ROOT / 'configs' / 'BRISC' / 'DRL' / _CONFIG_MAP[TUMOR_TYPE]))
cfg          = resolve_agent_config(cfg_full, AGENT_NAME)
baseline_cfg = load_config(str(PKG_ROOT / 'configs' / cfg['baseline_cfg_name']))

cfg['checkpoint_dir'] = str(DRIVE_CKPT_DIR)

# Auto-detect BRISC data root and U-Net checkpoint under Drive
brisc_roots = [p for p in DRIVE_DATA_ROOT.rglob('brisc2025') if p.is_dir()]
if not brisc_roots:
    brisc_roots = [p for p in DRIVE_DATA_ROOT.rglob('BRISC') if p.is_dir()]
if brisc_roots:
    baseline_cfg['data_root'] = str(brisc_roots[0])
else:
    print(f'[WARN] BRISC data root not found under {DRIVE_DATA_ROOT} - upload it to Drive first.')

ckpt_cands = [p for p in DRIVE_DATA_ROOT.rglob('brisc_best.pt')]
if ckpt_cands:
    cfg['baseline_checkpoint'] = str(ckpt_cands[0])
else:
    print(f'[WARN] brisc_best.pt not found under {DRIVE_DATA_ROOT} - upload the baseline checkpoint.')

print(f"Agent           : {cfg['agent_type']}")
print(f"Target class    : {cfg['target_class']} ({cfg['class_name']})")
print(f"Tumor filter    : {cfg.get('tumor_type_filter') or 'all (tumor vs non-tumor)'}")
print(f"Reward mode     : {cfg['reward_mode']}")
print(f"BRISC data root : {baseline_cfg.get('data_root')}")
print(f"Baseline ckpt   : {cfg.get('baseline_checkpoint')}")
print(f"Train steps     : {cfg['train_steps']}")
print(f"Checkpoint dir  : {cfg['checkpoint_dir']}")
print(f"Device          : {get_device()}")


## 2 · Warm-start — pre-compute U-Net init masks

Runs the U-Net over all splits once and caches `{image, gt_mask, init_mask, prob_map}` dicts.
The DRL training loop never touches the U-Net again.

**BRISC notes:**
- Images are resized from variable sizes (up to 624 px) to 256x256 by the transforms.
- z-score normalisation is applied (MRI standard - robust to scanner drift).
- `tumor_type_filter` (set automatically from your `TUMOR_TYPE` choice above) restricts
  warm-start to one tumour subtype when not `'all'`.

~8-15 min on T4 for the combined set (9 586 samples); proportionally faster per subtype.


In [ ]:
from iteris.warm_start import precompute_init_masks
import numpy as np

train_samples, val_samples, test_samples = precompute_init_masks(
    baseline_cfg        = baseline_cfg,
    baseline_checkpoint = cfg['baseline_checkpoint'],
    target_class        = cfg['target_class'],
    min_area_fraction   = cfg.get('min_area_fraction', 0.005),
    tumor_type_filter    = cfg.get('tumor_type_filter', None),
)

print(f"\nSamples - train: {len(train_samples)}  val: {len(val_samples)}  test: {len(test_samples)}")
if cfg.get('tumor_type_filter'):
    print(f"  (filtered to tumor_type = {cfg['tumor_type_filter']!r})")

def dice_to_iou(d): return d / (2.0 - d + 1e-8)

init_dices = [
    float((s['init_mask'] & s['gt_mask']).sum() * 2 /
          (s['init_mask'].sum() + s['gt_mask'].sum() + 1e-6))
    for s in val_samples
]
init_ious = [dice_to_iou(d) for d in init_dices]
gt_areas  = [s['gt_mask'].mean() * 100 for s in val_samples]

print(f"U-Net init Dice on val (tumor): mean {np.mean(init_dices):.4f}  "
      f"median {np.median(init_dices):.4f}")
print(f"U-Net init IoU  on val (tumor): mean {np.mean(init_ious):.4f}  "
      f"median {np.median(init_ious):.4f}")
print(f"GT tumour area  (val)         : mean {np.mean(gt_areas):.2f}%  "
      f"range [{min(gt_areas):.2f}%, {max(gt_areas):.2f}%]")


## 3 · OPTIONAL — dry-run sanity check (~2-3 min)

Runs a short training run on a small slice of train/val samples.
Self-contained `deepcopy(cfg)` — does NOT mutate `cfg`, `train_samples`, or `val_samples`.


In [ ]:
# ==============================================================================
RUN_DRY_RUN = False   # Set True for a quick sanity check before full training
# ==============================================================================

if RUN_DRY_RUN:
    import copy
    from iteris.drl_training import run_drl_training

    _dry = copy.deepcopy(cfg)
    _dry.update(dict(
        train_steps         = 600,
        prefill_steps       = 500,
        buffer_size         = 3000,
        eval_every          = 200,
        epsilon_decay_steps = 400,
        batch_size          = 32,
    ))
    _dry_result = run_drl_training(_dry, train_samples[:30], val_samples[:10])
    print(f"\nDry run passed. Best val score (meaningless at 600 steps): {_dry_result['best_dice']:.4f}")
    print(f"  cfg['train_steps'] still = {cfg['train_steps']}  (unchanged)")
    print('  -> Safe to run section 4 below for the real training run.')
else:
    print('Dry run skipped (RUN_DRY_RUN = False). Proceed directly to section 4.')


## 4 · Full training

| Agent | Est. time on T4 | Steps |
|---|---|---|
| DuelingDDQN | ~1.5-2 hr | 40 000-60 000 (subtype-dependent) |
| DDPG | ~5-6 hr | 100 000 |

**Free Colab note:** sessions disconnect after ~12 h or ~90 min idle. Checkpoints save to
Drive (`DRIVE_CKPT_DIR`) whenever val score improves, so a disconnect mid-run only costs the
time since the last improvement — the partially-trained checkpoint stays safe on Drive.


In [ ]:
from iteris.drl_training import run_drl_training

result    = run_drl_training(cfg, train_samples, val_samples)
agent     = result['agent']
history   = result['history']     # pandas DataFrame
best_dice = result['best_dice']

print(f"\nTraining complete - {cfg['agent_type']} on {cfg['class_name']}")
print(f"  Best val final-Dice : {best_dice:.4f}")
print(f"  Checkpoint          : {result['checkpoint']}")


## 5 · Visualisation setup

**BRISC-specific ENV_KW differences vs CAMUS:**
- `shift_px` is finer (small tumours need sub-pixel precision)
- `sdt_clip` is tighter (small structures have shorter signed-distance gradients)
- reward mode is `iou_delta` / `dice_pbrs` depending on the chosen config — HD95 is not
  part of the training reward but is computed separately in §6/§10 for display.


In [ ]:
from iteris.refinement_viz import (
    refinement_env_kwargs, build_replays, plot_comparison,
    plot_playback, plot_behaviour, evaluate_testset)

ENV_KW  = refinement_env_kwargs(cfg)
N_VIZ   = 8
replays = build_replays(agent, val_samples, ENV_KW, n_viz=N_VIZ, seed=0)

CLASS_NAME = cfg.get('class_name', '')
print(f'Built {len(replays)} greedy replays for visualisation.')


## 6 · Sample comparison — best / median / worst gains

In [ ]:
import matplotlib.pyplot as plt

fig = plot_comparison(
    replays, baseline_cfg, cfg,
    class_idx=cfg.get('target_class', 1), class_name=CLASS_NAME,
    out_path=str(DRIVE_CKPT_DIR / f"{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_comparison.png"))
plt.show()


## 7 · Iteration playback — all steps on best-gain sample

Rose overlay = agent mask · Green contour = GT boundary.


In [ ]:
fig = plot_playback(
    replays[-1], cfg, class_name=CLASS_NAME,
    out_path=str(DRIVE_CKPT_DIR / f"{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_playback.png"))
plt.show()


## 8 · Behaviour analysis — trajectories + action distribution

In [ ]:
fig = plot_behaviour(
    replays, cfg, class_name=CLASS_NAME,
    out_path=str(DRIVE_CKPT_DIR / f"{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_behaviour.png"))
plt.show()


## 9 · Learning curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['step'], history['init_dice_mean'],  label='Init Dice (U-Net)', ls='--', alpha=0.6, color='gray')
ax1.plot(history['step'], history['final_dice_mean'], label=f"Final Dice ({cfg['agent_type']})", color='C0')
if 'best_dice_mean' in history.columns:
    ax1.plot(history['step'], history['best_dice_mean'], label='Best-seen Dice (ceiling)', color='C2', alpha=0.7)
ax1.set(xlabel='Training step', ylabel='Mean Val Dice',
        title=f"{cfg['dataset']} - {cfg['class_name']} refinement curve")
ax1.legend(); ax1.grid(alpha=0.3)

delta = history['final_dice_mean'] - history['init_dice_mean']
ax2.plot(history['step'], delta, color='C0')
ax2.axhline(0, color='k', lw=0.8)
ax2.fill_between(history['step'], delta, 0, where=(delta > 0), alpha=0.15, color='green')
ax2.fill_between(history['step'], delta, 0, where=(delta < 0), alpha=0.15, color='red')
ax2.set(xlabel='Training step', ylabel='Delta Dice (final - init)',
        title=f"Refinement gain - {cfg['agent_type']} ({cfg['class_name']})")
ax2.grid(alpha=0.3)

plt.suptitle(f"{cfg['dataset']} {cfg['agent_type']} - {cfg['class_name']} learning curves")
plt.tight_layout()
out = str(DRIVE_CKPT_DIR / f"{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_curves.png")
plt.savefig(out, dpi=150); plt.show(); print(f'Saved to {out}')


## 10 · Test-set evaluation + summary JSON

Full test set evaluation with greedy policy. Computes Dice, IoU, and HD95 for comparison
against the U-Net baseline (test Dice 0.8351, HD95 8.36 px).


In [ ]:
import json
test_metrics = evaluate_testset(agent, test_samples, ENV_KW)
print(json.dumps(test_metrics, indent=2))

summary = {**test_metrics,
           'agent_type':    cfg['agent_type'],
           'target_class':  cfg['target_class'],
           'class_name':    cfg['class_name'],
           'dataset':       cfg['dataset'],
           'tumor_type':    TUMOR_TYPE,
           'paradigm':      'local_mask_refinement'}
out = str(DRIVE_CKPT_DIR / f"{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_test_results.json")
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved to', out)
print(f"\nBaseline (U-Net) Dice: {test_metrics['init_dice_mean']:.4f}")
print(f"Refined  (agent)  Dice: {test_metrics['final_dice_mean']:.4f}  "
      f"(Delta {test_metrics['delta_dice_mean']:+.4f})")
print(f"Best-seen ceiling Dice: {test_metrics['best_dice_mean']:.4f}")
